# Use case 6 — Querying instance data across studies

**Questions:**

1. *Corpus orientation.* Which studies are in the corpus, and what are
   their codes, titles, design types, and phases?
2. *Corpus census.* Which USDM classes do real E2J outputs actually
   instantiate, how often per study — decorated with `skos:definition`
   and NCIt anchors from `usdm_v4.ttl`, because model and data share
   IRIs.

**Corpus.** Real protocols from `data4knowledge/usdm_data`
(`source_data/protocols/`, pinned commit below): USDM Excel-to-JSON
(E2J) renderings of published study protocols — CDISC Pilot, Alexion
NCT04573309, Eli Lilly NCT03421379, Sanofi NCT03637764. A fifth file
(Roche NCT04320615) carries no `usdmVersion` envelope and is skipped by
the load gate. Instance test data, not a modelling source — the same
rule as the pilot fetch in `notebooks/30_validate.ipynb`.

**Design.** Each study loads into its own named graph. This works because
the instance context (decision D5) sets no `@base`: document-scoped ids
like `Activity_1` resolve against a per-file base IRI, so identical ids
in different studies cannot collide, and `GRAPH ?study` doubles as
provenance. The ontology itself is one more named graph in the same
Dataset. `07_validation_audit.ipynb` builds on the same corpus and adds
the SHACL layers.

Pinned to the v0.5.0 ontology baseline (8,642 triples). First run
fetches the study JSONs (network required once).


## Step 0 — fetch the corpus

Five candidate study JSONs from `data4knowledge/usdm_data`, pinned to
an explicit commit so reruns see identical bytes. SHA-256 sidecar:
`../downloads/d4k/.fetch_meta_examples.json`.


In [1]:
import hashlib
import json
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import rdflib
from rdflib.namespace import RDF

TTL_PATH = "../usdm_v4.ttl"
CONTEXT_PATH = "../usdm_v4.context.jsonld"
EXPECTED_BASELINE = 8642
USDM = "https://w3id.org/cdisc/usdm/v4/"

D4K_REPO = "data4knowledge/usdm_data"
D4K_COMMIT = "66cabf898f71f81b2bb2bf9bc80d09bd69eca68c"  # main, 2026-03-31
D4K_DIR = Path("../downloads/d4k")
D4K_DIR.mkdir(parents=True, exist_ok=True)

STUDY_PATHS = (
    "source_data/protocols/CDISC_Pilot/CDISC_Pilot_Study.json",
    "source_data/protocols/Alexion_NCT04573309_Wilsons/Alexion_NCT04573309_Wilsons.json",
    "source_data/protocols/EliLilly_NCT03421379_Diabetes/EliLilly_NCT03421379_Diabetes.json",
    "source_data/protocols/Sanofi_NCT03637764_Oncology/Sanofi_NCT03637764_Oncology.json",
    "source_data/protocols/Roche_NCT04320615_COVID/Roche_NCT04320615_COVID.json",
)

def raw_url(rel):
    return f"https://raw.githubusercontent.com/{D4K_REPO}/{D4K_COMMIT}/{rel}"

META_PATH = D4K_DIR / ".fetch_meta_examples.json"
meta = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}

fetched = 0
for rel in STUDY_PATHS:
    local = D4K_DIR / Path(rel).name
    if local.exists():
        continue
    with urllib.request.urlopen(raw_url(rel)) as resp:
        data = resp.read()
    local.write_bytes(data)
    meta[Path(rel).name] = {
        "commit": D4K_COMMIT,
        "raw_url": raw_url(rel),
        "sha256": hashlib.sha256(data).hexdigest(),
        "size_bytes": len(data),
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    fetched += 1
    print(f"fetched {Path(rel).parent.name}: {len(data):,} bytes")

if fetched:
    META_PATH.write_text(json.dumps(meta, indent=2) + "\n")
print(f"{fetched} fetched, {len(STUDY_PATHS) - fetched} already present")


0 fetched, 5 already present


## Step 1 — build the Dataset: one named graph per study, plus the model

Each study JSON is unwrapped from its API envelope, gated on
`usdmVersion == "4.0.0"` (the version this ontology renders), given the
instance context, and parsed into a named graph whose IRI is the file's
raw GitHub URL at the pinned tag — the graph name *is* the provenance.
`publicID` supplies the per-file base that keeps document-scoped ids
apart.


In [2]:
with open(CONTEXT_PATH) as f:
    context = json.load(f)["@context"]

ds = rdflib.Dataset()
MODEL_GRAPH = rdflib.URIRef(USDM)
model = ds.graph(MODEL_GRAPH)
model.parse(TTL_PATH, format="turtle")
print(f"model graph: {len(model)} triples")
if len(model) != EXPECTED_BASELINE:
    print(f"  note: triple count differs from v0.5.0 baseline ({EXPECTED_BASELINE})"
          " — likely a DDF-RA tag bump; this notebook is pinned to v0.5.0 anchor shape")

def short(uri):
    """Last path segment, fragment-aware."""
    if uri is None:
        return None
    s = str(uri)
    if "#" in s:
        return s.split("#")[-1]
    return s.split("/")[-1]

graph_key = {}
rows = []
for rel in STUDY_PATHS:
    key = Path(rel).parent.name
    local = D4K_DIR / Path(rel).name
    doc = json.loads(local.read_text())
    usdm_version = doc.get("usdmVersion")
    if usdm_version != "4.0.0":
        print(f"skipping {key}: usdmVersion={usdm_version!r}, need '4.0.0'")
        continue
    study = doc["study"]
    study["@context"] = context
    url = raw_url(rel)
    g = ds.graph(rdflib.URIRef(url))
    g.parse(data=json.dumps(study), format="json-ld", publicID=url)
    graph_key[url] = key
    rows.append({
        "study": key,
        "system": f"{doc.get('systemName')} {doc.get('systemVersion')}",
        "triples": len(g),
        "typed_nodes": len(set(g.subjects(RDF.type, None))),
    })

corpus_df = pd.DataFrame(rows)
print(f"corpus: {len(corpus_df)} studies, {corpus_df['triples'].sum():,} instance triples")
corpus_df


model graph: 8642 triples
skipping Roche_NCT04320615_COVID: usdmVersion=None, need '4.0.0'
corpus: 4 studies, 43,100 instance triples


,study,system,triples,typed_nodes
0,CDISC_Pilot,CDISC USDM E2J 0.67.0,12274,2013
1,Alexion_NCT04573309_Wilsons,CDISC USDM E2J 0.67.0,10619,1666
2,EliLilly_NCT03421379_Diabetes,CDISC USDM E2J 0.67.0,11070,1908
3,Sanofi_NCT03637764_Oncology,CDISC USDM E2J 0.67.0,9137,1409


,study,system,triples,typed_nodes
0,CDISC_Pilot,CDISC USDM E2J 0.67.0,12274,2013
1,Alexion_NCT04573309_Wilsons,CDISC USDM E2J 0.67.0,10619,1666
2,EliLilly_NCT03421379_Diabetes,CDISC USDM E2J 0.67.0,11070,1908
3,Sanofi_NCT03637764_Oncology,CDISC USDM E2J 0.67.0,9137,1409


## Step 2 — the simplest possible query: one row per study

Before anything clever: `?study` is the named graph (the file's raw
GitHub URL at the pinned commit, shortened to its protocol subfolder
name for display),
and everything else comes from inside the study. `Study.name` is the
study code the sponsor put in the JSON.


In [3]:
BASIC_SPARQL = """
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?study ?name ?title ?versionIdentifier
WHERE {
  GRAPH ?study {
    ?s a usdm:Study ;
       usdm:Study-name ?name .
    OPTIONAL {
      ?s usdm:Study-versions/usdm:StudyVersion-titles ?t .
      ?t usdm:StudyTitle-text ?title ;
         usdm:StudyTitle-type/usdm:Code-decode ?titleType .
      FILTER(CONTAINS(LCASE(STR(?titleType)), "brief"))
    }
    OPTIONAL {
      ?s usdm:Study-versions/usdm:StudyVersion-versionIdentifier ?versionIdentifier .
    }
  }
  FILTER(?study != <https://w3id.org/cdisc/usdm/v4/>)
}
"""

rows = []
for r in ds.query(BASIC_SPARQL):
    rows.append({
        "study": graph_key[str(r["study"])],
        "study_code": str(r["name"]),
        "brief_title": str(r["title"]) if r["title"] is not None else None,
        "version": str(r["versionIdentifier"]) if r["versionIdentifier"] is not None else None,
    })

basic_df = pd.DataFrame(rows).drop_duplicates().sort_values("study").reset_index(drop=True)
basic_df


,study,study_code,brief_title,version
0,Alexion_NCT04573309_Wilsons,ALXN1840-WD-204,Copper and Molybdenum Balance in Participants ...,2
1,CDISC_Pilot,CDISC PILOT - LZZT,Xanomeline (LY246708),2
2,EliLilly_NCT03421379_Diabetes,LY900018,A Study of Nasal Glucagon (LY900018) in Japane...,1
3,Sanofi_NCT03637764_Oncology,Test NCT 03637764,"Safety, preliminary efficacy and PK of isatuxi...",amendment 5


## Step 3 — corpus orientation

One query over all named graphs: study name, official title, design
type, phase. Titles hang off `StudyVersion.titles` with a typed `Code`;
phase is an `AliasCode` on the study design, read through its
`standardCode`.


In [4]:
ORIENTATION_SPARQL = """
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>

SELECT ?study ?studyName ?title ?design ?phase
WHERE {
  GRAPH ?study {
    ?s a usdm:Study ;
       usdm:Study-name ?studyName .
    OPTIONAL {
      ?s usdm:Study-versions/usdm:StudyVersion-titles ?t .
      ?t usdm:StudyTitle-text ?title ;
         usdm:StudyTitle-type/usdm:Code-decode ?titleType .
      FILTER(CONTAINS(LCASE(STR(?titleType)), "official"))
    }
    OPTIONAL {
      ?s usdm:Study-versions/usdm:StudyVersion-studyDesigns ?d .
      ?d a ?design .
      OPTIONAL {
        ?d usdm:StudyDesign-studyPhase/usdm:AliasCode-standardCode/usdm:Code-decode ?phase .
      }
    }
  }
  FILTER(?study != <https://w3id.org/cdisc/usdm/v4/>)
}
"""

rows = []
for r in ds.query(ORIENTATION_SPARQL):
    rows.append({
        "study": graph_key[str(r["study"])],
        "study_name": str(r["studyName"]),
        "official_title": str(r["title"]) if r["title"] is not None else None,
        "design_type": short(r["design"]),
        "phase": str(r["phase"]) if r["phase"] is not None else None,
    })

orientation_df = pd.DataFrame(rows).drop_duplicates().sort_values("study").reset_index(drop=True)
orientation_df


,study,study_name,official_title,design_type,phase
0,Alexion_NCT04573309_Wilsons,ALXN1840-WD-204,"A Phase 2, Open-label Study to Assess Copper a...",InterventionalStudyDesign,Phase II Trial
1,CDISC_Pilot,CDISC PILOT - LZZT,Safety and Efficacy of the Xanomeline Transder...,InterventionalStudyDesign,Phase II Trial
2,EliLilly_NCT03421379_Diabetes,LY900018,A Phase 3 Study of Nasal Glucagon (LY900018) C...,InterventionalStudyDesign,Phase III Trial
3,Sanofi_NCT03637764_Oncology,Test NCT 03637764,"A Phase 1/2 open-label, multi-center, safety, ...",InterventionalStudyDesign,Phase I/II Trial


## Step 4 — corpus census with the model joined in

Which classes does real instance data exercise, and how often? One
query counts instances per class per study graph and decorates each
class with `skos:definition` and the paired NCIt anchors (decision D4)
from the model graph. Model and data share IRIs, so the join needs no
mapping table — just a second `GRAPH` clause.


In [5]:
MODEL_JOIN_SPARQL = """
PREFIX usdm: <https://w3id.org/cdisc/usdm/v4/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?cls ?study (COUNT(?inst) AS ?n) ?definition ?evs ?obo
WHERE {
  GRAPH ?study { ?inst a ?cls . }
  FILTER(?study != <https://w3id.org/cdisc/usdm/v4/>)
  GRAPH <https://w3id.org/cdisc/usdm/v4/> {
    OPTIONAL { ?cls skos:definition ?definition . }
    OPTIONAL {
      ?cls skos:exactMatch ?evs .
      FILTER(STRSTARTS(STR(?evs), "http://ncicb.nci.nih.gov/"))
    }
    OPTIONAL {
      ?cls skos:exactMatch ?obo .
      FILTER(STRSTARTS(STR(?obo), "http://purl.obolibrary.org/obo/NCIT_"))
    }
  }
}
GROUP BY ?cls ?study ?definition ?evs ?obo
"""

rows = []
for r in ds.query(MODEL_JOIN_SPARQL):
    rows.append({
        "class": short(r["cls"]),
        "study": graph_key[str(r["study"])],
        "instances": int(r["n"]),
        "definition": str(r["definition"]) if r["definition"] is not None else None,
        "ncit_evs": short(r["evs"]),
        "ncit_obo": str(r["obo"]) if r["obo"] is not None else None,
    })

census_long = pd.DataFrame(rows)
index_cols = ["class", "definition", "ncit_evs", "ncit_obo"]
census_long[index_cols] = census_long[index_cols].fillna("")
census_df = (
    census_long
    .pivot_table(index=index_cols, columns="study", values="instances",
                 aggfunc="sum", fill_value=0)
    .reset_index()
)
census_df.columns.name = None
census_df["total"] = census_df[list(graph_key.values())].sum(axis=1)
census_df = census_df.sort_values("total", ascending=False).reset_index(drop=True)
print(f"{len(census_df)} of 80 concrete classes instantiated in the corpus")
with pd.option_context("display.max_colwidth", 55):
    display(census_df.head(15))


64 of 80 concrete classes instantiated in the corpus


,class,definition,ncit_evs,ncit_obo,Alexion_NCT04573309_Wilsons,CDISC_Pilot,EliLilly_NCT03421379_Diabetes,Sanofi_NCT03637764_Oncology,total
0,Code,A symbol or combination of symbols which is assigne...,C25162,http://purl.obolibrary.org/obo/NCIT_C25162,622,681,640,433,2376
1,AliasCode,An alternative symbol or combination of symbols whi...,C201344,http://purl.obolibrary.org/obo/NCIT_C201344,111,246,270,86,713
2,NarrativeContent,The container that holds an instance of unstructure...,C207592,http://purl.obolibrary.org/obo/NCIT_C207592,105,231,112,113,561
3,ResponseCode,A symbol or combination of symbols representing the...,C201347,http://purl.obolibrary.org/obo/NCIT_C201347,88,236,185,42,551
4,BiomedicalConceptProperty,A characteristic from a set of characteristics used...,C202493,http://purl.obolibrary.org/obo/NCIT_C202493,80,197,217,55,549
5,NarrativeContentItem,An individual item within the container that holds ...,C215489,http://purl.obolibrary.org/obo/NCIT_C215489,105,86,112,113,416
6,Timing,The chronological relationship between temporal eve...,C80484,http://purl.obolibrary.org/obo/NCIT_C80484,66,24,35,57,182
7,ScheduledActivityInstance,A scheduled occurrence of an activity event.,C201350,http://purl.obolibrary.org/obo/NCIT_C201350,65,24,35,52,176
8,Activity,"An action, undertaking, or event, which is anticipa...",C71473,http://purl.obolibrary.org/obo/NCIT_C71473,44,36,34,54,168
9,EligibilityCriterion,Characteristics which are necessary to allow a subj...,C16112,http://purl.obolibrary.org/obo/NCIT_C16112,31,31,36,60,158


## Takeaways

- **The named-graph design carries the whole use case.** One Dataset,
  five graphs: 43,100 instance triples from four studies plus the 8,642
  model triples. Identical document-scoped ids (`Activity_1` exists in
  every study) never collide because D5 leaves the base to the loader,
  and `GRAPH ?study` gives every result row its provenance. The load
  gate earns its place on the first run: the Roche file carries no
  `usdmVersion` envelope and is skipped, visibly.
- **Even the simplest query surfaces identity quirks:** Sanofi's
  `Study.name` is "Test NCT 03637764" — payload study codes are
  test-tagged or reused, which is why the named graph (the pinned
  source file), not `Study.name`, is the reliable study key.
- **The census shows how much of USDM v4 real E2J outputs exercise**,
  and the model join costs nothing: definitions and paired NCIt
  anchors (D4) decorate the counts because model and data share IRIs —
  no mapping table, no lookup service, just a second `GRAPH` clause.
- For validation of this corpus — both SHACL layers, findings
  decorated the same way — see `07_validation_audit.ipynb`.
